In [37]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI
import json
from IPython.display import Markdown,display, update_display

In [17]:
load_dotenv(override=True)

model_name = os.getenv('MODEL_NAME')
base_url = os.getenv('OLLAMA_BASE_URL')

if not model_name and not base_url:
  print("No MODEL_NAME and OLLAMA_BASE_URL found!")
else:
  print("MODEL_NAME and OLLAMA_BASE_URL found!, It looks good so far!")

client = OpenAI(
  base_url=base_url,
  api_key="ollama"
)

MODEL_NAME and OLLAMA_BASE_URL found!, It looks good so far!


In [18]:
links = fetch_website_links("https://www.srmmachinecenter.com/")
links

['/',
 '/',
 '/',
 '/company-profile.html',
 '/products.html',
 'drilling-tapping-machine.html',
 '/38mm-pillar-drilling-machine-9280073.html',
 '/geared-radial-drill-machine-9280074.html',
 '/19mm-pillar-drilling-machine-9280075.html',
 '/25-mm-pillar-drilling-machine-9280076.html',
 '/radial-drilling-machine-9280077.html',
 '/12mm-tapping-machine-9280078.html',
 '/drilling-cum-tapping-machine-9280079.html',
 '/19-mm-drilling-and-tapping-machine-9280080.html',
 '/6mm-vertical-tapping-machine-9280081.html',
 '/industrial-drilling-and-tapping-machine-9280082.html',
 '/milling-cum-drilling-machine-9280083.html',
 '/magnetic-drilling-machine-9280084.html',
 '/25mm-radial-drilling-machine-9280085.html',
 '/magnetic-core-drilling-machine-9280086.html',
 'milling-machine.html',
 '/vertical-turret-milling-machine-9280087.html',
 '/turret-milling-machine-9280088.html',
 '/industrial-milling-machine-9280089.html',
 '/dro-milling-machine-9280090.html',
 '/horizontal-milling-machine-9280091.html'

In [19]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [20]:
def get_links_user_prompt(url):
  user_prompt = f"""
    Here is the list of links on the website {url} -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links):
  """
  links = fetch_website_links(url)
  user_prompt += "\n".join(links)
  return user_prompt

In [21]:
print(get_links_user_prompt("https://www.srmmachinecenter.com/"))


    Here is the list of links on the website https://www.srmmachinecenter.com/ -
    Please decide which of these are relevant web links for a brochure about the company, 
    respond with the full https URL in JSON format.
    Do not include Terms of Service, Privacy, email links.

    Links (some might be relative links):
  /
/
/
/company-profile.html
/products.html
drilling-tapping-machine.html
/38mm-pillar-drilling-machine-9280073.html
/geared-radial-drill-machine-9280074.html
/19mm-pillar-drilling-machine-9280075.html
/25-mm-pillar-drilling-machine-9280076.html
/radial-drilling-machine-9280077.html
/12mm-tapping-machine-9280078.html
/drilling-cum-tapping-machine-9280079.html
/19-mm-drilling-and-tapping-machine-9280080.html
/6mm-vertical-tapping-machine-9280081.html
/industrial-drilling-and-tapping-machine-9280082.html
/milling-cum-drilling-machine-9280083.html
/magnetic-drilling-machine-9280084.html
/25mm-radial-drilling-machine-9280085.html
/magnetic-core-drilling-machine-928008

In [ ]:
def select_relevant_links(url):
  response = client.chat.completions.create(
    model=model_name,
    messages=[
      {"role": "system", "content": link_system_prompt},
      {"role": "user", "content": get_links_user_prompt(url)}
      ],
      response_format={"type": "json_object"}
  )
  result = response.choices[0].message.content
  links = json.loads(result)
  return links

In [27]:
select_relevant_links("https://www.srmmachinecenter.com/")

{'links': [{'type': 'company profile',
   'url': 'https://www.srmmachinecenter.com/company-profile.html'}]}

In [47]:
def fetch_page_all_relevant_links(url):
  contents = fetch_website_contents(url)
  relevant_links = select_relevant_links(url)
  result = f'## Landing Page:\n\n{contents}\n## Relevant Links:\n"'

  for link in relevant_links['links']:
    result += f"\n\n### Link: {link['type']}\n"
    result += fetch_website_contents(link["url"])
  return result

In [48]:
print(fetch_page_all_relevant_links("https://www.srmmachinecenter.com/"))

## Landing Page:

Geared Radial Drill Machine & Industrial Drilling Machine Manufacturer in Chennai

Talk to us
07971459475
SRM MACHINE CENTER
GST : 33BWHPS7160K1Z3
Home Page
Company Profile
Our Products
Drilling And Tapping Machine
38mm Pillar Drilling Machine
Geared Radial Drill Machine
19mm Pillar Drilling Machine
25 mm Pillar Drilling Machine
Radial Drilling Machine
12mm Tapping Machine
Drilling Cum Tapping Machine
19 mm Drilling And Tapping Machine
6mm Vertical Tapping Machine
Industrial Drilling And Tapping Machine
Milling Cum Drilling Machine
Magnetic Drilling Machine
25mm Radial Drilling Machine
Magnetic Core Drilling Machine
Milling Machine
Vertical Turret Milling Machine
Turret Milling Machine
Industrial Milling Machine
Dro Milling Machine
Horizontal Milling Machine
Ram Turret Milling Machine
Vertical Lifting Table Milling Machine
Lathe Machine
Light Duty Lathe Machine
7 Feet Belt Driven Heavy Duty Lathe Machine
Medium Duty Lathe Machine
9 Feet Cone Pulley And Belt Driven Hea

In [31]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [32]:
def get_brochure_user_prompt(company_name, url):
   user_prompt = f"""
    You are looking at a company called: {company_name}
    Here are the contents of its landing page and other relevant pages;
    use this information to build a short brochure of the company in markdown without code blocks.\n\n
  """
   user_prompt += fetch_page_all_relevant_links(url)
   user_prompt = user_prompt[:5_000]
   return user_prompt

In [33]:
get_brochure_user_prompt("SRM MACHINE CENTER", "https://www.srmmachinecenter.com/")

'\n    You are looking at a company called: SRM MACHINE CENTER\n    Here are the contents of its landing page and other relevant pages;\n    use this information to build a short brochure of the company in markdown without code blocks.\n\n\n  ## Landing Page:\n\nGeared Radial Drill Machine & Industrial Drilling Machine Manufacturer in Chennai\n\nTalk to us\n07971459475\nSRM MACHINE CENTER\nGST : 33BWHPS7160K1Z3\nHome Page\nCompany Profile\nOur Products\nDrilling And Tapping Machine\n38mm Pillar Drilling Machine\nGeared Radial Drill Machine\n19mm Pillar Drilling Machine\n25 mm Pillar Drilling Machine\nRadial Drilling Machine\n12mm Tapping Machine\nDrilling Cum Tapping Machine\n19 mm Drilling And Tapping Machine\n6mm Vertical Tapping Machine\nIndustrial Drilling And Tapping Machine\nMilling Cum Drilling Machine\nMagnetic Drilling Machine\n25mm Radial Drilling Machine\nMagnetic Core Drilling Machine\nMilling Machine\nVertical Turret Milling Machine\nTurret Milling Machine\nIndustrial Mill

In [35]:
def create_brochure(company_name, url):
  response = client.chat.completions.create(
    model=model_name,
    messages=[
      {"role": "system", "content": brochure_system_prompt},
      {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
      ]
  )
  result = response.choices[0].message.content
  display(Markdown(result))

In [38]:
create_brochure("SRM MACHINE CENTER", "https://www.srmmachinecenter.com/")

# SRM MACHINE CENTER

## Overview
SRM MACHINE CENTER is a leading manufacturer of drilling and tapping machines, milling machines, lathe machines, power presses, grinding machines, and more. Based in Chennai, Tamil Nadu, we offer a wide range of high-quality industrial equipment to meet the diverse needs of our customers in the manufacturing sector.

### Contact Information:
- **Phone:** 07971459475
- **GST No.:** 33BWHPS7160K1Z3

## Our Products

### Drilling and Tapping Machines:
- **Geared Radial Drill Machine**
- **38mm Pillar Drilling Machine**
- **19mm Pillar Drilling Machine**
- **25mm Pillar Drilling Machine**
- **Radial Drilling Machine**

### Milling Machines:
- **3 Jaw Chuck**
- **Milling Machine Spares Parts**
- **Milling Machine Spindle**
- **Dro Digital Readout Milling System**

### Lathe Machines:
- **Light Duty Lathe Machine**
- **7 Feet Belt Driven Heavy Duty Lathe Machine**

### Power Press Machines:
- **C Type Power Press**
- **20 Ton C Type Power Press**
- **Pneumatic Power Press**

## Company Profile
SRM MACHINE CENTER has a strong commitment to quality and customer satisfaction. Our experienced team of professionals works diligently to ensure that all products meet or exceed industry standards. We prioritize the maintenance and repair of industrial machinery, providing a full range of spare parts for our machines.

## Career Opportunities
At SRM MACHINE CENTER, we are always looking for talented individuals who share our passion for precision and innovation. Whether you's seeking roles in engineering, sales, manufacturing, operations or客户服务, consider joining us today!

- **Engineering**
- **Sales**
- **Manufacturing**
- **Operations**
- **Customer Service**

## Get In Touch
For any inquiries or to place an order, please don't hesitate to contact us directly via email at `info@srmmachinecenter.com` or reach out by phone to `07971459475`.

---

Join SRM MACHINE CENTER today and be part of a committed team that delivers excellence in industrial machinery solutions!

In [40]:
def steam_brochure(company_name, url):
  stream = client.chat.completions.create(
    model=model_name,
    messages=[
        {"role": "system", "content": brochure_system_prompt},
        {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
      ],
    stream=True
  )
  response = ""
  display_handle = display(Markdown(""), display_id=True)
  for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)

In [41]:
steam_brochure("SRM MACHINE CENTER", "https://www.srmmachinecenter.com/")

---

# SRM MACHINE CENTER   

### Manufacturer, Supplier, and Trader of Precision Machinery in Chennai

Welcome to SRM MACHINE CENTER, your premier destination for quality equipment. We specialize in the manufacturing, supply, and trade of a wide range of drilling machines and milling equipment designed to meet diverse industrial needs.

## Key Features:

- **Geared Radial Drill Machines**: Perfect for detailed work with precise accuracy.
- **Cutting Tools & Presses**: Offered in various sizes, from 3mm up to powerful 10 ton hydraulic models.
- **Lathe Machines**: Including manual and hydraulic options for extensive applications.
- **Milling Machines and Turret Milling Machines**: Tailored for precision machining with various vertical or horizontal configurations.

## Why Choose SRM MACHINE CENTER?

### High-Quality Products
Our machinery is meticulously engineered and built to the highest standards, ensuring durability and reliability. We use only original parts and components to maintain quality.

### Experienced Team
SRM MACHINE CENTER boasts a knowledgeable team dedicated to providing exceptional support and assistance. Our experts can help you optimize your production processes and choose equipment that best fits your requirements.

### Competitive Pricing
We pride ourselves on offering competitive pricing with no compromise on quality or service. Contact us today to explore our range of products!

#### Contact Us:
- **Phone:** 07971459475  
- **GST No.:** 33BWHPS7160K1Z3  
---

### Our Products

SRM MACHINE CENTER offers a comprehensive range of products, including:

- **Drilling and Tapping Machines**: Various models for industrial grade work.
- **Milling Machines**: From vertical to horizontal configurations for varied applications.
- **Lathe Machines**: Light-duty and heavy-duty options for precision machining.
- **Power Press Machines**: Different sizes catering to various production needs.

For detailed product information, visit our [products page](products-page-url) or contact us directly.

---

[Back to Home Page]

---